In [70]:
# ═══════════════════════════════════════════
# Cell 1 — Load dataset from Google Drive
# ═══════════════════════════════════════════
import pandas as pd

df = pd.read_csv('balanced_veremi_dataset.csv')


print(f"Dataset loaded: {df.shape}")
print(df.columns.tolist())
print(df['AttackerType'].value_counts())

Dataset loaded: (2269540, 19)
['rcvTime', 'pos_0', 'pos_1', 'pos_noise_0', 'pos_noise_1', 'spd_0', 'spd_1', 'spd_noise_0', 'spd_noise_1', 'acl_0', 'acl_1', 'acl_noise_0', 'acl_noise_1', 'hed_0', 'hed_1', 'hed_noise_0', 'hed_noise_1', 'ReceiverID', 'AttackerType']
AttackerType
DoSRandomSybil        113477
RandomSpeed           113477
RandomSpeedOffset     113477
GridSybil             113477
DataReplay            113477
ConstPosOffset        113477
DoSDisruptive         113477
DelayedMessages       113477
Benign                113477
ConstSpeed            113477
ConstSpeedOffset      113477
DataReplaySybil       113477
DoS                   113477
ConstPos              113477
DoSRandom             113477
RandomPos             113477
EventualStop          113477
RandomPosOffset       113477
DoSDisruptiveSybil    113477
Disruptive            113477
Name: count, dtype: int64


In [71]:
# ═══════════════════════════════════════════
# Cell 2 — Install dependencies
# ═══════════════════════════════════════════
!pip install xgboost lightgbm scikit-learn pandas numpy joblib -q
print("All packages ready")

All packages ready


In [72]:
# CELL 3
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import joblib
import json

# Save original FIRST before any modification
df_original = df.copy()
print(f"Original saved: {df_original.shape}")
print(f"Benign: {(df_original['AttackerType']=='Benign').sum()}")
print(f"Attack: {(df_original['AttackerType']!='Benign').sum()}")

# Encode ReceiverID
le_receiver = LabelEncoder()
df['ReceiverID'] = le_receiver.fit_transform(df['ReceiverID'])
df_original['ReceiverID'] = le_receiver.transform(df_original['ReceiverID'])
joblib.dump(le_receiver, 'receiver_encoder.pkl')

# Binary Y
df['Y_binary'] = df['AttackerType'].apply(lambda x: 0 if x == 'Benign' else 1)
df_original['Y_binary'] = df_original['AttackerType'].apply(lambda x: 0 if x == 'Benign' else 1)

# Multiclass Y
le = LabelEncoder()
df['Y_multi'] = le.fit_transform(df['AttackerType'])
df_original['Y_multi'] = le.transform(df_original['AttackerType'])

joblib.dump(le, 'label_encoder.pkl')
label_mapping = {str(i): label for i, label in enumerate(le.classes_)}
with open('label_mapping.json', 'w') as f:
    json.dump(label_mapping, f, indent=2)

print(f"\ndf_original Benign: {(df_original['Y_binary']==0).sum()}")
print(f"df_original Attack: {(df_original['Y_binary']==1).sum()}")
print(f"Attack types: {len(le.classes_)}")

Original saved: (2269540, 19)
Benign: 113477
Attack: 2156063

df_original Benign: 113477
df_original Attack: 2156063
Attack types: 20


In [73]:
# CELL 3C — Delta features + sliding window features
print("=== Adding delta + window features ===")

# Work on attack-only for multiclass
df_attack_only = df[df['Y_binary'] == 1].copy()
df_attack_only = df_attack_only.sort_values(
    ['ReceiverID', 'rcvTime']).reset_index(drop=True)

# Delta features
df_attack_only['delta_time']  = df_attack_only.groupby('ReceiverID')['rcvTime'].diff()
df_attack_only['delta_pos_0'] = df_attack_only.groupby('ReceiverID')['pos_0'].diff()
df_attack_only['delta_pos_1'] = df_attack_only.groupby('ReceiverID')['pos_1'].diff()
df_attack_only['delta_spd_0'] = df_attack_only.groupby('ReceiverID')['spd_0'].diff()
df_attack_only['delta_hed_0'] = df_attack_only.groupby('ReceiverID')['hed_0'].diff()
df_attack_only['delta_dist']  = (
    df_attack_only['delta_pos_0']**2 +
    df_attack_only['delta_pos_1']**2)**0.5
df_attack_only['expected_dist'] = (
    df_attack_only['spd_0'].abs() *
    df_attack_only['delta_time'].abs())
df_attack_only['dist_error'] = (
    df_attack_only['delta_dist'] -
    df_attack_only['expected_dist'])
df_attack_only['delta_pos_noise_0'] = df_attack_only.groupby(
    'ReceiverID')['pos_noise_0'].diff()
df_attack_only['delta_pos_noise_1'] = df_attack_only.groupby(
    'ReceiverID')['pos_noise_1'].diff()

df_attack_only = df_attack_only.dropna().reset_index(drop=True)

# Sliding window features (last WINDOW messages per vehicle)
# These capture temporal patterns invisible to per-message features.
WINDOW = 10

df_attack_only['pos_std'] = (
    df_attack_only.groupby('ReceiverID')['pos_0']
    .transform(lambda x: x.rolling(WINDOW, min_periods=1).std().fillna(0)))

df_attack_only['spd_std'] = (
    df_attack_only.groupby('ReceiverID')['spd_0']
    .transform(lambda x: x.rolling(WINDOW, min_periods=1).std().fillna(0)))

df_attack_only['hed_std'] = (
    df_attack_only.groupby('ReceiverID')['hed_0']
    .transform(lambda x: x.rolling(WINDOW, min_periods=1).std().fillna(0)))

# msg_rate = 1 / mean(delta_time) over WINDOW msgs — high for DoS/Disruptive
df_attack_only['msg_rate'] = (
    df_attack_only.groupby('ReceiverID')['delta_time']
    .transform(lambda x: 1.0 / (x.rolling(WINDOW, min_periods=1).mean().abs() + 1e-9)))

print(f"Attack-only with delta + window features: {df_attack_only.shape}")
print(df_attack_only['AttackerType'].value_counts())

=== Adding delta + window features ===
Attack-only with delta + window features: (2156061, 35)
AttackerType
Disruptive            113477
RandomPosOffset       113477
DoSRandom             113477
DoS                   113477
GridSybil             113477
ConstSpeedOffset      113477
DoSRandomSybil        113477
ConstPos              113477
DataReplay            113477
DataReplaySybil       113477
EventualStop          113477
DoSDisruptive         113477
RandomPos             113477
DelayedMessages       113477
ConstPosOffset        113477
ConstSpeed            113477
DoSDisruptiveSybil    113477
RandomSpeed           113476
RandomSpeedOffset     113476
Name: count, dtype: int64


In [74]:
# CELL 3B — Class grouping + balanced multiclass dataset
print("=== Class grouping + balanced datasets ===")

# 8 → 5 groups
# FloodAttack + SybilAttack → DoSFamily   (indistinguishable from single-receiver features)
# PositionFreeze + PositionSpoof → PositionAttack  (both report wrong position, model confused)
ATTACK_GROUPS = {
    'DoS':                'DoSFamily',
    'DoSRandom':          'DoSFamily',
    'DoSDisruptive':      'DoSFamily',
    'Disruptive':         'DoSFamily',
    'DoSRandomSybil':     'DoSFamily',
    'DoSDisruptiveSybil': 'DoSFamily',
    'DataReplaySybil':    'DoSFamily',
    'GridSybil':          'DoSFamily',
    'ConstPos':           'PositionAttack',
    'ConstPosOffset':     'PositionAttack',
    'RandomPos':          'PositionAttack',
    'RandomPosOffset':    'PositionAttack',
    'ConstSpeed':         'SpeedManip',
    'ConstSpeedOffset':   'SpeedManip',
    'RandomSpeed':        'SpeedManip',
    'RandomSpeedOffset':  'SpeedManip',
    'DataReplay':         'ReplayAttack',
    'DelayedMessages':    'DelayedMessages',
    'EventualStop':       'EventualStop',
}

df_attack_only['AttackGroup'] = df_attack_only['AttackerType'].map(ATTACK_GROUPS)
print("\nClass distribution after grouping:")
print(df_attack_only['AttackGroup'].value_counts())

# ── Binary dataset (unchanged) ──
df_benign_orig = df_original[df_original['Y_binary'] == 0]
df_attack_orig = df_original[df_original['Y_binary'] == 1]
n_benign = len(df_benign_orig)
df_attack_sample = df_attack_orig.sample(n=n_benign, random_state=42)
df_binary = pd.concat([df_benign_orig, df_attack_sample])
df_binary = df_binary.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"\nBinary dataset: {len(df_binary)} rows  (Benign {(df_binary['Y_binary']==0).sum()} / Attack {(df_binary['Y_binary']==1).sum()})")

# ── Multiclass dataset — balanced by AttackGroup ──
n_per_group = df_attack_only['AttackGroup'].value_counts().min()
print(f"\nSamples per group (capped at min): {n_per_group:,}")

balanced = []
for group in sorted(df_attack_only['AttackGroup'].unique()):
    subset = df_attack_only[df_attack_only['AttackGroup'] == group]
    balanced.append(subset.sample(n=n_per_group, random_state=42))
    print(f"  {group:<20} — {n_per_group:,}")

df_multi = pd.concat(balanced).sample(frac=1, random_state=42).reset_index(drop=True)

le_multi = LabelEncoder()
df_multi['Y_multi'] = le_multi.fit_transform(df_multi['AttackGroup'])
joblib.dump(le_multi, 'label_encoder_multi.pkl')

label_mapping_multi = {str(i): label for i, label in enumerate(le_multi.classes_)}
with open('label_mapping_multi.json', 'w') as f:
    json.dump(label_mapping_multi, f, indent=2)

print(f"\nMulticlass dataset: {len(df_multi):,} rows — {df_multi['AttackGroup'].nunique()} groups")
print("Label mapping:", label_mapping_multi)

=== Class grouping + balanced datasets ===

Class distribution after grouping:
AttackGroup
DoSFamily          907816
PositionAttack     453908
SpeedManip         453906
EventualStop       113477
DelayedMessages    113477
ReplayAttack       113477
Name: count, dtype: int64

Binary dataset: 226954 rows  (Benign 113477 / Attack 113477)

Samples per group (capped at min): 113,477
  DelayedMessages      — 113,477
  DoSFamily            — 113,477
  EventualStop         — 113,477
  PositionAttack       — 113,477
  ReplayAttack         — 113,477
  SpeedManip           — 113,477

Multiclass dataset: 680,862 rows — 6 groups
Label mapping: {'0': 'DelayedMessages', '1': 'DoSFamily', '2': 'EventualStop', '3': 'PositionAttack', '4': 'ReplayAttack', '5': 'SpeedManip'}


In [75]:
# Cell 4 — Train/test split + scale
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib

EXCLUDE = {'AttackerType', 'AttackGroup', 'Y_binary', 'Y_multi', 'ReceiverID'}

BINARY_COLS = [c for c in df_binary.columns if c not in EXCLUDE]
MULTI_COLS  = [c for c in df_multi.columns  if c not in EXCLUDE]

print(f"Binary features  ({len(BINARY_COLS)}): {BINARY_COLS}")
print(f"Multiclass features ({len(MULTI_COLS)}): {MULTI_COLS}")

X_b = df_binary[BINARY_COLS]
Y_b = df_binary['Y_binary']
X_m = df_multi[MULTI_COLS]
Y_m = df_multi['Y_multi']

X_train_b, X_test_b, yb_train, yb_test = train_test_split(
    X_b, Y_b, test_size=0.2, random_state=42, stratify=Y_b)
X_train_m, X_test_m, ym_train, ym_test = train_test_split(
    X_m, Y_m, test_size=0.2, random_state=42, stratify=Y_m)

scaler_binary = StandardScaler()
X_train_b_scaled = scaler_binary.fit_transform(X_train_b)
X_test_b_scaled  = scaler_binary.transform(X_test_b)
joblib.dump(scaler_binary, 'scaler_binary.pkl')

scaler_multi = StandardScaler()
X_train_m_scaled = scaler_multi.fit_transform(X_train_m)
X_test_m_scaled  = scaler_multi.transform(X_test_m)
joblib.dump(scaler_multi, 'scaler_multi.pkl')

print(f"\nBinary    — Train: {len(X_train_b)} | Test: {len(X_test_b)}")
print(f"Multiclass — Train: {len(X_train_m)} | Test: {len(X_test_m)}")
print(f"Num classes: {Y_m.nunique()}")
print("Scalers saved.")

Binary features  (17): ['rcvTime', 'pos_0', 'pos_1', 'pos_noise_0', 'pos_noise_1', 'spd_0', 'spd_1', 'spd_noise_0', 'spd_noise_1', 'acl_0', 'acl_1', 'acl_noise_0', 'acl_noise_1', 'hed_0', 'hed_1', 'hed_noise_0', 'hed_noise_1']
Multiclass features (31): ['rcvTime', 'pos_0', 'pos_1', 'pos_noise_0', 'pos_noise_1', 'spd_0', 'spd_1', 'spd_noise_0', 'spd_noise_1', 'acl_0', 'acl_1', 'acl_noise_0', 'acl_noise_1', 'hed_0', 'hed_1', 'hed_noise_0', 'hed_noise_1', 'delta_time', 'delta_pos_0', 'delta_pos_1', 'delta_spd_0', 'delta_hed_0', 'delta_dist', 'expected_dist', 'dist_error', 'delta_pos_noise_0', 'delta_pos_noise_1', 'pos_std', 'spd_std', 'hed_std', 'msg_rate']

Binary    — Train: 181563 | Test: 45391
Multiclass — Train: 544689 | Test: 136173
Num classes: 6
Scalers saved.


In [76]:
# ═══════════════════════════════════════════
# Cell 5 — Train XGBoost Binary
# ═══════════════════════════════════════════
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score)
import time

print("=== Training XGBoost Binary Classifier ===")
t0 = time.time()

xgb_binary = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    use_label_encoder=False,
    eval_metric='logloss',
    verbosity=1
)

xgb_binary.fit(
    X_train_b_scaled, yb_train,
    eval_set=[(X_test_b_scaled, yb_test)],
    verbose=50
)

train_time  = time.time() - t0
pred_binary = xgb_binary.predict(X_test_b_scaled)

acc       = accuracy_score(yb_test, pred_binary)
f1        = f1_score(yb_test, pred_binary)
cm        = confusion_matrix(yb_test, pred_binary)
tn, fp, fn, tp = cm.ravel()
fpr       = fp / (fp + tn) if (fp + tn) > 0 else 0
fnr       = fn / (fn + tp) if (fn + tp) > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0

print(f"\n=== Binary Results ===")
print(f"Accuracy:            {acc:.4f}")
print(f"F1 Score:            {f1:.4f}")
print(f"Precision:           {precision:.4f}")
print(f"Recall:              {recall:.4f}")
print(f"False Positive Rate: {fpr:.4f}")
print(f"False Negative Rate: {fnr:.4f}")
print(f"Training time:       {train_time:.1f}s")
print(f"\nConfusion Matrix:")
print(f"  TN={tn}  FP={fp}")
print(f"  FN={fn}  TP={tp}")
print(classification_report(yb_test, pred_binary,
      target_names=['Benign', 'Attack']))

joblib.dump(xgb_binary, 'model_binary.pkl')
print("Binary model saved")

=== Training XGBoost Binary Classifier ===
[0]	validation_0-logloss:0.59816
[50]	validation_0-logloss:0.00299


/opt/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [20:00:17] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[100]	validation_0-logloss:0.00004
[150]	validation_0-logloss:0.00001
[199]	validation_0-logloss:0.00001

=== Binary Results ===
Accuracy:            1.0000
F1 Score:            1.0000
Precision:           1.0000
Recall:              1.0000
False Positive Rate: 0.0000
False Negative Rate: 0.0000
Training time:       0.7s

Confusion Matrix:
  TN=22696  FP=0
  FN=0  TP=22695
              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00     22696
      Attack       1.00      1.00      1.00     22695

    accuracy                           1.00     45391
   macro avg       1.00      1.00      1.00     45391
weighted avg       1.00      1.00      1.00     45391

Binary model saved


In [79]:
# Cell 6 — Train multiclass: XGBoost + LightGBM
import xgboost as xgb
from xgboost import XGBClassifier
import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import time

num_classes = Y_m.nunique()
group_names = le_multi.classes_
print(f"=== Multiclass Training — {num_classes} groups ===\n")

# ── XGBoost ───────────────────────────────────────────────────────────────────
print("--- XGBoost ---")
t0 = time.time()

xgb_multi = XGBClassifier(
    n_estimators=5000,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0.1,
    random_state=42,
    n_jobs=-1,
    objective='multi:softprob',
    num_class=num_classes,
    eval_metric='mlogloss',
    verbosity=1,
    callbacks=[xgb.callback.EarlyStopping(rounds=100, save_best=True)],
)

xgb_multi.fit(
    X_train_m_scaled, ym_train,
    eval_set=[(X_test_m_scaled, ym_test)],
    verbose=200,
)

pred_xgb = xgb_multi.predict(X_test_m_scaled)
acc_xgb  = accuracy_score(ym_test, pred_xgb)
f1_xgb   = f1_score(ym_test, pred_xgb, average='weighted')
print(f"\nXGBoost  — Accuracy: {acc_xgb:.4f}  F1: {f1_xgb:.4f}  Time: {time.time()-t0:.0f}s")
print(classification_report(ym_test, pred_xgb, target_names=group_names))

# ── LightGBM ──────────────────────────────────────────────────────────────────
print("\n--- LightGBM ---")
t0 = time.time()

lgbm_multi = LGBMClassifier(
    n_estimators=5000,
    max_depth=8,
    learning_rate=0.05,
    num_leaves=127,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=20,
    random_state=42,
    n_jobs=-1,
    objective='multiclass',
    num_class=num_classes,
    verbose=-1,
)

lgbm_multi.fit(
    X_train_m_scaled, ym_train,
    eval_set=[(X_test_m_scaled, ym_test)],
    callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(200)],
)

pred_lgbm = lgbm_multi.predict(X_test_m_scaled)
acc_lgbm  = accuracy_score(ym_test, pred_lgbm)
f1_lgbm   = f1_score(ym_test, pred_lgbm, average='weighted')
print(f"\nLightGBM — Accuracy: {acc_lgbm:.4f}  F1: {f1_lgbm:.4f}  Time: {time.time()-t0:.0f}s")
print(classification_report(ym_test, pred_lgbm, target_names=group_names))

# ── Save the better model ──────────────────────────────────────────────────────
best_model = lgbm_multi if acc_lgbm >= acc_xgb else xgb_multi
best_name  = 'LightGBM' if acc_lgbm >= acc_xgb else 'XGBoost'
joblib.dump(best_model, 'model_multiclass.pkl')
print(f"\nSaved {best_name} as model_multiclass.pkl  (accuracy {max(acc_xgb, acc_lgbm):.4f})")

=== Multiclass Training — 6 groups ===

--- XGBoost ---
[0]	validation_0-mlogloss:1.77098
[200]	validation_0-mlogloss:1.39304
[400]	validation_0-mlogloss:1.30940
[600]	validation_0-mlogloss:1.25383
[800]	validation_0-mlogloss:1.21402
[1000]	validation_0-mlogloss:1.18219
[1200]	validation_0-mlogloss:1.15526
[1400]	validation_0-mlogloss:1.13259
[1600]	validation_0-mlogloss:1.11284
[1800]	validation_0-mlogloss:1.09548
[2000]	validation_0-mlogloss:1.07954
[2200]	validation_0-mlogloss:1.06412
[2400]	validation_0-mlogloss:1.05129
[2600]	validation_0-mlogloss:1.03944
[2800]	validation_0-mlogloss:1.02919
[3000]	validation_0-mlogloss:1.01908
[3200]	validation_0-mlogloss:1.00945
[3400]	validation_0-mlogloss:1.00093
[3600]	validation_0-mlogloss:0.99259
[3800]	validation_0-mlogloss:0.98479
[4000]	validation_0-mlogloss:0.97693
[4200]	validation_0-mlogloss:0.97009
[4400]	validation_0-mlogloss:0.96387
[4600]	validation_0-mlogloss:0.95791
[4800]	validation_0-mlogloss:0.95231
[4999]	validation_0-mloglo